In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
LOG_DIR = "/content/drive/MyDrive/Research_Logs"

print(f"✅ ログ保存先: {LOG_DIR}")

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)

%pip install rank_bm25
%pip install xopen

from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime
import xopen
import random

random.seed(0)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

from ScalingController import ScalingController
from get_model_safe_last_attnweight import get_model_safe, test_patched_model_sanity, remove_cache
from main_inference import main_inference
from run_eval import run_eval_exact_match
from analyze_log import ExperimentAnalyzer

# if 'CACHED_MODEL' not in globals():
#     CACHED_MODEL = None
#     CACHED_TOKENIZER = None

def confirmation():
    """実行前にユーザーの確認を求める"""
    print("\n" + "!" * 30)
    print("警告: 書き出しモードが 'a' (追記) です。")
    print("既存のファイルにデータが追加されますが、よろしいですか？")
    print("!" * 30 + "\n")

    answer = input("実行する場合は 'yes' と入力してください: ").strip().lower()

    if answer != 'yes':
        print("\n[中止] 実行がキャンセルされました。")
        # Jupyterでセルを止める最もクリーンな方法
        raise KeyboardInterrupt("User cancelled the execution.")

    print("\n[開始] 実験を実行します...\n")

Mounted at /content/drive
✅ ログ保存先: /content/drive/MyDrive/Research_Logs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 15.0 MB/s eta 0:00:00


In [2]:
from datasets.formatting.formatting import TypeVar
from typing import List, Optional, Type
from pydantic.dataclasses import dataclass

import json
import torch
import gc
import os
from tqdm import tqdm
from xopen import xopen
from copy import deepcopy
from itertools import islice
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM


PROMPTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/lost_in_the_middle/prompts")
T = TypeVar("T")

@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        isgold = data.pop("isgold", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score, isgold=isgold))

@dataclass
class Document:
    title: str
    text: str
    @classmethod
    def from_dict(cls, data):
        return cls(title=data.get("title", ""), text=data.get("text", ""))

# --- 推論・プロンプト関数 ---
def get_qa_prompt(question: str, documents: List[Document]):
    # FORMAT: Document [1](Title: ...) ...
    formatted_documents = []
    for i, doc in enumerate(documents):
        formatted_documents.append(f"Document [{i+1}](Title: {doc.title}) {doc.text}")

    # 標準的なQAプロンプトテンプレート
    prompt_template = "Answer the question based on the following documents.\n\n{search_results}\n\nQuestion: {question}\nAnswer:"
    return prompt_template.format(question=question, search_results="\n".join(formatted_documents))

def run_batch_inference(prompts: List[str], model, tokenizer):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    input_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.0,
            do_sample=False
        )

    # バッチ全体の生成テキストをデコード
    results = []
    for i in range(len(prompts)):
        generated_tokens = outputs[i][input_len:]
        text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(text.strip())
    return results


In [3]:
# --- メイン処理 ---
def main():
    # モデルの準備
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        use_cache=True,
        trust_remote_code=True
    )
    LIMIT = 600

    done_count = 0
    if os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, "r") as f:
            done_count = sum(1 for _ in f)

    # すでに上限に達している場合は終了
    if done_count >= LIMIT:
        print(f"Already processed {done_count} items. Limit is {LIMIT}.")
        return

    with xopen(NQ_path, "r") as f_data, xopen(OUTPUT_PATH, "a") as f_out:
        # 2. isliceの第3引数にLIMITを指定（done_countからLIMITまで）
        # これにより、すでに終わった分を除いて、合計600件になるまで読み込みます
        f_data_skipped = islice(f_data, done_count, LIMIT)

        # tqdmの総数も制限に合わせて設定
        pbar = tqdm(total=LIMIT, initial=done_count, desc="Processing Batches")

        while True:
            # BATCH_SIZE分だけ読み込むが、islice(f_data_skipped, ...) なので
            # LIMITに達した瞬間に lines_data が空になります
            lines_data = list(islice(f_data_skipped, BATCH_SIZE))

            # (lines_attnも同様にisliceを使っている場合は、同様に制限が必要です)
            # もし lines_attn もあるなら、同様に islice(f_attn, done_count, LIMIT) を作成してください

            if not lines_data:
                break

            batch_prompts = []
            batch_original_data = []

            for l_data in lines_data: # l_attnとのzipは必要に応じて修正してください
                data = json.loads(l_data)
                docs = [Document.from_dict(ctx) for ctx in data["ctxs"]]
                prompt = get_qa_prompt(data["question"], docs)
                batch_prompts.append(prompt)
                batch_original_data.append(data)

            # バッチ推論
            predictions = run_batch_inference(batch_prompts, model, tokenizer)

            # 結果の書き出し
            for original_data, pred in zip(batch_original_data, predictions):
                output_obj = deepcopy(original_data)
                output_obj["prediction"] = pred
                f_out.write(json.dumps(output_obj) + "\n")

            f_out.flush()
            pbar.update(len(lines_data))

            # メモリ解放
            gc.collect()
            torch.cuda.empty_cache()

    print(f"\nProcessing complete. Results saved to: {OUTPUT_PATH}")

In [ ]:
# --- shift ---
MODEL = "vicuna"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "shift_bm25"
BATCH_SIZE = 16

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/check_vicuna-16k_0_index.jsonl"


if __name__ == "__main__":
    main()

tokenizer_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

Processing Batches: 100%|██████████| 600/600 [11:52<00:00,  1.19s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/check_vicuna-16k_0_index.jsonl
